# 3 · De palavras a significado: uma ontologia da cafeteria
**Café IA · Floripa | 15 minutos**

Uma ontologia define **classes**, **relações** e **regras de significado** para um domínio.
Um grafo registra fatos usando esse vocabulário. Vamos armazenar os dois no mesmo Autonomous
usado pela memória e inferir fatos que não foram escritos explicitamente.

Implementação didática: triplas `(sujeito, predicado, objeto)` em uma tabela Oracle e uma regra
de herança de tipos escrita em SQL. O vocabulário usa RDF/RDFS/OWL, mas este exemplo não instala
nem afirma executar um reasoner OWL completo. Ao final mostramos a extensão para RDF Graph nativo.
Não há chamada a LLM: toda conclusão tem uma trilha de fatos verificável no banco.

## 3.1 · O vocabulário antes dos dados

| Elemento | Exemplo | Significado |
|---|---|---|
| Classe | `cafe:CafeComLeite` | Tipo de bebida |
| Subclasse | `CafeComLeite → Bebida` | Todo café com leite é uma bebida |
| Indivíduo | `cafe:CappuccinoDaCasa` | Uma opção específica do cardápio |
| Relação | `cafe:temIngrediente` | Liga bebida a ingrediente |
| Tipo do ingrediente | `cafe:IngredienteAnimal` | Classificação explícita do ingrediente |

Caminho que queremos descobrir:
`CappuccinoDaCasa → CaféComLeite → Bebida` e
`CappuccinoDaCasa → temIngrediente → LeiteVaca → IngredienteAnimal`.
Isso permite perguntar por bebidas com ingrediente animal mesmo quando o cardápio não usa essas palavras.

In [1]:
import pandas as pd
from IPython.display import display
from apoio import conectar

conexao = conectar()
def consultar(sql, **parametros):
    with conexao.cursor() as cursor:
        cursor.execute(sql, parametros)
        return pd.DataFrame(cursor.fetchall(), columns=[d[0] for d in cursor.description])

display(consultar("select sys_context('USERENV','DB_NAME') BANCO from dual"))

,BANCO
0,GA66906C10B87CD_AMDEMOERP


## 3.2 · Criar uma tabela exclusiva para a demo
A chave composta evita duplicatas. A criação é idempotente: se a tabela já existe, validamos
suas colunas antes de inserir. Não há `DROP`, `TRUNCATE` nem alteração de tabelas de aplicações.
`origem` separa o vocabulário ontológico dos fatos do cardápio.

In [2]:
with conexao.cursor() as cursor:
    cursor.execute("select count(*) from user_tables where table_name='CAFE_ONT_TRIPLAS'")
    if cursor.fetchone()[0] == 0:
        cursor.execute('''
            CREATE TABLE CAFE_ONT_TRIPLAS (
                sujeito VARCHAR2(120) NOT NULL,
                predicado VARCHAR2(120) NOT NULL,
                objeto VARCHAR2(120) NOT NULL,
                origem VARCHAR2(16) NOT NULL,
                CONSTRAINT CAFE_ONT_TRIPLAS_PK PRIMARY KEY (sujeito, predicado, objeto),
                CONSTRAINT CAFE_ONT_ORIGEM_CK CHECK (origem IN ('ontologia', 'cardapio'))
            )
        ''')
    cursor.execute('select sujeito, predicado, objeto, origem from CAFE_ONT_TRIPLAS where 1=0')
print('Tabela pronta.')

Tabela pronta.


## 3.3 · Inserir a ontologia e os fatos
`rdf:type` afirma o tipo de um recurso; `rdfs:subClassOf` estabelece herança.
`rdfs:domain` e `rdfs:range` documentam os tipos esperados da relação.
Aqui a regra SQL implementará somente herança de tipos, não todas as inferências de domain/range.
Os nomes com prefixo são CURIEs; os namespaces completos estão em `ontologia_cafe.ttl`.

In [3]:
ontologia = [
    ('cafe:Bebida', 'rdf:type', 'owl:Class'),
    ('cafe:CafeComLeite', 'rdf:type', 'owl:Class'),
    ('cafe:CafeComLeite', 'rdfs:subClassOf', 'cafe:Bebida'),
    ('cafe:Ingrediente', 'rdf:type', 'owl:Class'),
    ('cafe:IngredienteAnimal', 'rdf:type', 'owl:Class'),
    ('cafe:IngredienteAnimal', 'rdfs:subClassOf', 'cafe:Ingrediente'),
    ('cafe:Laticinio', 'rdf:type', 'owl:Class'),
    ('cafe:Laticinio', 'rdfs:subClassOf', 'cafe:IngredienteAnimal'),
    ('cafe:IngredienteVegetal', 'rdf:type', 'owl:Class'),
    ('cafe:IngredienteVegetal', 'rdfs:subClassOf', 'cafe:Ingrediente'),
    ('cafe:temIngrediente', 'rdf:type', 'owl:ObjectProperty'),
    ('cafe:temIngrediente', 'rdfs:domain', 'cafe:Bebida'),
    ('cafe:temIngrediente', 'rdfs:range', 'cafe:Ingrediente'),
]
cardapio = [
    ('cafe:CappuccinoDaCasa', 'rdf:type', 'cafe:CafeComLeite'),
    ('cafe:CappuccinoDaCasa', 'cafe:temIngrediente', 'cafe:LeiteVaca'),
    ('cafe:CappuccinoAveia', 'rdf:type', 'cafe:CafeComLeite'),
    ('cafe:CappuccinoAveia', 'cafe:temIngrediente', 'cafe:LeiteAveia'),
    ('cafe:Espresso', 'rdf:type', 'cafe:Bebida'),
    ('cafe:LeiteVaca', 'rdf:type', 'cafe:Laticinio'),
    ('cafe:LeiteAveia', 'rdf:type', 'cafe:IngredienteVegetal'),
]
with conexao.cursor() as cursor:
    for origem, triplas in [('ontologia', ontologia), ('cardapio', cardapio)]:
        cursor.executemany('''
            MERGE INTO CAFE_ONT_TRIPLAS t
            USING (SELECT :1 sujeito, :2 predicado, :3 objeto, :4 origem FROM dual) n
            ON (t.sujeito=n.sujeito AND t.predicado=n.predicado AND t.objeto=n.objeto)
            WHEN NOT MATCHED THEN INSERT (sujeito,predicado,objeto,origem)
                VALUES (n.sujeito,n.predicado,n.objeto,n.origem)
        ''', [(*tripla, origem) for tripla in triplas])
conexao.commit()
display(consultar('select * from CAFE_ONT_TRIPLAS order by origem,sujeito,predicado'))

,SUJEITO,PREDICADO,OBJETO,ORIGEM
0,cafe:CappuccinoAveia,cafe:temIngrediente,cafe:LeiteAveia,cardapio
1,cafe:CappuccinoAveia,rdf:type,cafe:CafeComLeite,cardapio
2,cafe:CappuccinoDaCasa,cafe:temIngrediente,cafe:LeiteVaca,cardapio
3,cafe:CappuccinoDaCasa,rdf:type,cafe:CafeComLeite,cardapio
4,cafe:Espresso,rdf:type,cafe:Bebida,cardapio
5,cafe:LeiteAveia,rdf:type,cafe:IngredienteVegetal,cardapio
6,cafe:LeiteVaca,rdf:type,cafe:Laticinio,cardapio
7,cafe:Bebida,rdf:type,owl:Class,ontologia
8,cafe:CafeComLeite,rdf:type,owl:Class,ontologia
9,cafe:CafeComLeite,rdfs:subClassOf,cafe:Bebida,ontologia


## 3.4 · Uma consulta literal perde significado
Leite de vaca está explicitamente classificado como `Laticinio`. Procurar somente a string
`IngredienteAnimal` nos tipos declarados do cardápio não encontra nenhum indivíduo.
Precisamos seguir as relações entre as classes.

In [4]:
literal = consultar('''
    SELECT sujeito, objeto FROM CAFE_ONT_TRIPLAS
    WHERE origem='cardapio' AND predicado='rdf:type' AND objeto=:classe
''', classe='cafe:IngredienteAnimal')
display(literal)
assert literal.empty

,SUJEITO,OBJETO


## 3.5 · Inferir pela hierarquia, dentro do Oracle
Regra usada: **se X é do tipo A e A é subclasse de B, então X é do tipo B**.
`CONNECT BY NOCYCLE` percorre as subclasses, incluindo mais de um nível.
`UNION` reúne tipos declarados e inferidos. Não reescrevemos os fatos originais.

In [5]:
TIPOS_SQL = '''
    WITH hierarquia AS (
        SELECT CONNECT_BY_ROOT sujeito AS classe, objeto AS ancestral
        FROM (
            SELECT sujeito, objeto FROM CAFE_ONT_TRIPLAS
            WHERE predicado='rdfs:subClassOf'
        )
        CONNECT BY NOCYCLE PRIOR objeto = sujeito
    ), tipos AS (
        SELECT sujeito AS entidade, objeto AS classe
        FROM CAFE_ONT_TRIPLAS WHERE predicado='rdf:type'
        UNION
        SELECT t.sujeito, h.ancestral
        FROM CAFE_ONT_TRIPLAS t JOIN hierarquia h ON t.objeto=h.classe
        WHERE t.predicado='rdf:type'
    )
'''
inferidos = consultar(TIPOS_SQL + '''
    SELECT entidade, classe FROM tipos
    WHERE entidade IN ('cafe:LeiteVaca', 'cafe:CappuccinoDaCasa')
    ORDER BY entidade, classe
''')
display(inferidos)
assert ((inferidos.ENTIDADE == 'cafe:LeiteVaca') &
        (inferidos.CLASSE == 'cafe:IngredienteAnimal')).any()
assert ((inferidos.ENTIDADE == 'cafe:CappuccinoDaCasa') &
        (inferidos.CLASSE == 'cafe:Bebida')).any()

,ENTIDADE,CLASSE
0,cafe:CappuccinoDaCasa,cafe:Bebida
1,cafe:CappuccinoDaCasa,cafe:CafeComLeite
2,cafe:LeiteVaca,cafe:Ingrediente
3,cafe:LeiteVaca,cafe:IngredienteAnimal
4,cafe:LeiteVaca,cafe:Laticinio


## 3.6 · Pergunta de negócio e trilha de evidências
**Quais bebidas têm um ingrediente de origem animal?**
A consulta cruza o tipo inferido da bebida, a relação declarada com o ingrediente e o tipo
inferido desse ingrediente. Não depende de o nome da bebida mencionar leite ou origem animal.

In [6]:
resposta = consultar(TIPOS_SQL + '''
    SELECT DISTINCT r.sujeito AS bebida, r.objeto AS ingrediente,
           i.classe AS classificacao_ingrediente
    FROM CAFE_ONT_TRIPLAS r
    JOIN tipos b ON b.entidade=r.sujeito AND b.classe='cafe:Bebida'
    JOIN tipos i ON i.entidade=r.objeto AND i.classe=:classe
    WHERE r.predicado='cafe:temIngrediente'
    ORDER BY bebida
''', classe='cafe:IngredienteAnimal')
display(resposta)
assert 'cafe:CappuccinoDaCasa' in resposta.BEBIDA.values
assert 'cafe:CappuccinoAveia' not in resposta.BEBIDA.values

evidencia = consultar('''
    SELECT sujeito, predicado, objeto FROM CAFE_ONT_TRIPLAS
    WHERE sujeito IN ('cafe:CappuccinoDaCasa','cafe:LeiteVaca',
                      'cafe:Laticinio','cafe:IngredienteAnimal','cafe:CafeComLeite')
    ORDER BY sujeito, predicado
''')
display(evidencia)

,BEBIDA,INGREDIENTE,CLASSIFICACAO_INGREDIENTE
0,cafe:CappuccinoDaCasa,cafe:LeiteVaca,cafe:IngredienteAnimal


,SUJEITO,PREDICADO,OBJETO
0,cafe:CafeComLeite,rdf:type,owl:Class
1,cafe:CafeComLeite,rdfs:subClassOf,cafe:Bebida
2,cafe:CappuccinoDaCasa,cafe:temIngrediente,cafe:LeiteVaca
3,cafe:CappuccinoDaCasa,rdf:type,cafe:CafeComLeite
4,cafe:IngredienteAnimal,rdf:type,owl:Class
5,cafe:IngredienteAnimal,rdfs:subClassOf,cafe:Ingrediente
6,cafe:Laticinio,rdf:type,owl:Class
7,cafe:Laticinio,rdfs:subClassOf,cafe:IngredienteAnimal
8,cafe:LeiteVaca,rdf:type,cafe:Laticinio


## 3.7 · Ausência de um fato não prova sua negação
Cappuccino de aveia não apareceu na resposta, mas isso **não prova que é vegano**:
não declaramos que a lista de ingredientes é completa nem regras sobre contaminação.
A ontologia registra o conhecimento disponível; ela não inventa os fatos que faltam.

Isso é especialmente importante ao combinar ontologia com IA: o LLM pode explicar a trilha
de evidências, mas deve manter a conclusão limitada aos fatos e regras realmente avaliados.

**Exercício:** adicione uma nova bebida que usa `LeiteVaca` e execute novamente as consultas.
Não acrescente uma classificação `IngredienteAnimal` diretamente: deixe a regra inferir.

## 3.8 · Como evoluir para RDF Graph nativo
Este SQL implementa uma regra pequena e inspecionável. Para uma ontologia maior, use o arquivo
Turtle fornecido como ponto de partida e avalie RDF Graph, SPARQL e inferência RDFS/OWL do Oracle.
A configuração de rede semântica e eventuais requisitos de JVM precisam ser preparados conforme
a versão e o serviço. Este notebook não habilita componentes nem modifica essa configuração.

Referências: [tecnologias semânticas Oracle](https://docs.oracle.com/en/database/oracle/oracle-database/26/rdfrm/introduction-oracle-semantic-technologies-support.html),
[RDF no Autonomous](https://docs.oracle.com/en/database/oracle/oracle-database/26/rdfrm/quick-start-using-rdf-data-oracle-autonomous-ai-database.html),
[OWL e inferência](https://docs.oracle.com/en/database/oracle/oracle-database/26/rdfrm/using-owl-inferencing.html).

In [7]:
conexao.close()
print('Conexão encerrada. Ontologia e cardápio preservados no Autonomous.')

Conexão encerrada. Ontologia e cardápio preservados no Autonomous.
